# Wound-healing phenotype preprocessing

This notebook prepares the ear-hole wound-healing phenotype dataset used in the downstream statistical analyses.

The workflow:
1. loads the experimental sheet and genotype information;
2. removes failed or incomplete experiments;
3. propagates cage-mate genotype information where appropriate;
4. cleans and merges the ImageJ ear-hole measurements with metadata; and
5. exports the processed, untransformed phenotype table.

All paths are defined relative to the project directory so the notebook can be run on another computer without editing personal absolute paths.


## 1. Imports and project paths

Edit the directory names below only if your local repository uses a different layout.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

import seaborn as sns
import scipy.stats as stats
from scipy.stats import mannwhitneyu
from scipy.stats import shapiro, kstest
from scipy.stats import levene
from scipy.stats import ttest_ind

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm


# -----------------------------------------------------------------------------
# Project paths
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENTAL_SHEET_FILE = RAW_DATA_DIR / "experimental_sheet.csv"
FAILED_EXPERIMENTS_FILE = RAW_DATA_DIR / "failed_experiments.txt"
GENOTYPES_FILE = RAW_DATA_DIR / "genotypes.xlsx"
EAR_HOLE_AREA_FILE = RAW_DATA_DIR / "WH_area.xlsx"

EAR_HOLE_OUTPUT_CSV = PROCESSED_DATA_DIR / "ear_hole_area_untransformed.csv"
EAR_HOLE_OUTPUT_XLSX = PROCESSED_DATA_DIR / "ear_hole_area_untransformed.xlsx"


## 2. Prepare experimental metadata

Load cage, animal, and sex information, remove experiments listed as failed, remove incomplete rows, and retain the experimental records relevant to this study.


In [ ]:
experimental_sheet = pd.read_csv(
    EXPERIMENTAL_SHEET_FILE,
    usecols=['Cage', 'Platform ID', 'Our ID', 'Sex']
)

with open(FAILED_EXPERIMENTS_FILE, 'r') as file:
    exclude_exps = [line.strip() for line in file if line.strip()]

experimental_sheet_one = experimental_sheet[~experimental_sheet['Cage'].isin(exclude_exps)]
experimental_sheet_two = experimental_sheet_one.dropna(how='any')
experimental_sheet_three = experimental_sheet_two[
    experimental_sheet_two['Platform ID'].astype(str).str.startswith('2')
].copy()


### Add genotype information

Genotypes are mapped to animals using `Our ID`.


In [ ]:
genotypes = pd.read_excel(GENOTYPES_FILE)

lookup_dict = genotypes.set_index('Our ID')['Genotype'].to_dict()
experimental_sheet_three['Genotype'] = experimental_sheet_three['Our ID'].map(lookup_dict)


### Remove unmatched experimental pairs

Rows corresponding to `E` identifiers with no genotype information are identified, and the corresponding paired row is also removed.


In [ ]:
checks1 = experimental_sheet_three[experimental_sheet_three['Genotype'].isna() & experimental_sheet_three['Our ID'].astype(str).str.startswith('E')].index
print(checks1)

rows_to_drop = list(checks1) + [i + 1 for i in checks1 if i + 1 < len(experimental_sheet_three)]

# Step 3: Drop those rows from the original DataFrame
experimental_cleaned = experimental_sheet_three.drop(rows_to_drop).reset_index(drop=True)


### Complete paired genotype information

For every second row, a missing genotype is filled from the preceding paired animal. The number of replacements is reported as a check.


In [ ]:
replaced_count = 0

# Loop through every second row starting at index 1
for i in range(1, len(experimental_cleaned), 2):
    if pd.isna(experimental_cleaned.at[i, 'Genotype']):
        experimental_cleaned.at[i, 'Genotype'] = experimental_cleaned.at[i - 1, 'Genotype']
        replaced_count += 1

print(f"Number of NaNs replaced from the previous row: {replaced_count}")


### Check remaining missing genotype values


In [ ]:
nan_percent = experimental_cleaned['Genotype'].isna().mean() * 100
print(f"Percentage of missing genotype values: {nan_percent:.2f}%")


## 3. Prepare ear-hole measurements

Load the ImageJ-derived ear-hole measurements, retain valid measurements, extract the measured side, and standardize animal identifiers.


In [ ]:
ear_hole_area = pd.read_excel(EAR_HOLE_AREA_FILE)
ear_hole_area = ear_hole_area[ear_hole_area['was open at punching?'].isna()].reset_index(drop=True)
ear_hole_area['Side'] = ear_hole_area['ID'].str.extract(r'_([LR])')

# Remove the side suffix (_L or _R) from the animal ID.
ear_hole_area['ID'] = ear_hole_area['ID'].str.replace(r'[_LR]', '', regex=True)
ear_hole_area


### Merge phenotype measurements with genotype and sex metadata


In [ ]:
ear_hole_area = ear_hole_area.merge(
    experimental_cleaned[['Our ID', 'Genotype', 'Sex']], 
    left_on='ID', 
    right_on='Our ID', 
    how='left'
)

# Drop the redundant 'Our ID' column after the merge if you don't need it
ear_hole_area = ear_hole_area.drop(columns=['Our ID'])

ear_hole_area['Genotype'] = ear_hole_area['ID'].map(
    experimental_cleaned.set_index('Our ID')['Genotype']
)
ear_hole_area


### Remove non-study identifiers and missing area measurements


In [ ]:
ear_hole_area = ear_hole_area[~ear_hole_area['ID'].str.startswith('E')].reset_index(drop=True)
ear_hole_area = ear_hole_area.dropna(subset=['Area']).reset_index(drop=True)


### Retain animals with genotype information


In [ ]:
ear_hole_area_three = ear_hole_area[ear_hole_area['Genotype'].notna()].copy()


### Add genetic background (`COT`)

The first character of the animal ID is mapped to the corresponding mouse genetic background.


In [ ]:
ear_hole_area_three['COT'] = ear_hole_area_three['ID'].str[0].map({
    'M': 'MRL',
    'D': 'DBA',
    'B': 'B6',
    'C': 'CBA'
})


## 4. Export the untransformed phenotype dataset

This is the untransformed ear-hole phenotype table that can be used as the starting point for downstream statistical analysis.


In [ ]:
ear_hole_area_three.to_csv(EAR_HOLE_OUTPUT_CSV, index=False)
ear_hole_area_three.to_excel(EAR_HOLE_OUTPUT_XLSX, index=False)

print(f"Saved CSV:  {EAR_HOLE_OUTPUT_CSV}")
print(f"Saved XLSX: {EAR_HOLE_OUTPUT_XLSX}")
